# Phase 3 — Baseline M1: delete the object's Gaussians, measure the residual footprint
Selects the object's Gaussians two ways (geometric ground truth vs. 2D-mask lifting),
deletes them, renders held-out views, and scores against exact clean plates.
Runtime: GPU. After the install cell: restart, re-run cells 1–2, skip install.

In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Restore dataset + Phase-2 checkpoint at their original paths; rebuild splits
import os, glob, shutil, json
DRIVE = '/content/drive/MyDrive/light-footprint-removal'
DATA = '/content/data/footprint'
if not os.path.exists(f'{DATA}/transforms.json'):
    shutil.rmtree(DATA, ignore_errors=True)
    shutil.copytree(f'{DRIVE}/renders/footprint_dataset', DATA)
if not glob.glob('/content/outputs/**/config.yml', recursive=True):
    shutil.rmtree('/content/outputs', ignore_errors=True)
    shutil.copytree(f'{DRIVE}/checkpoints/phase2_with/outputs', '/content/outputs')
for f in glob.glob('/content/outputs/**/step-000015000.ckpt', recursive=True):
    os.remove(f)  # stale edited checkpoint from a previous run

root = f'{DATA}/with'
meta = json.load(open(f'{DATA}/transforms.json'))
frames = meta['frames']
test_idx = set(range(0, len(frames), 8))
for name, fr in {'train': [f for i, f in enumerate(frames) if i not in test_idx],
                 'test':  [f for i, f in enumerate(frames) if i in test_idx],
                 'val':   [f for i, f in enumerate(frames) if i in test_idx]}.items():
    json.dump({'camera_angle_x': meta['camera_angle_x'], 'frames': fr},
              open(f'{root}/transforms_{name}.json', 'w'))
print('frames:', len(os.listdir(f'{root}/rgb')), '| masks:', len(os.listdir(f'{root}/mask')))

In [ ]:
# Install. numpy is pinned LAST to 2.0.2: the checkpoint was saved under numpy 2.x
# and its pickles require it. Restart afterwards, re-run cells 1-2, skip this cell.
import os
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'
!pip -q install nerfstudio
!pip -q install --force-reinstall "numpy==2.0.2" 

In [ ]:
# Open the checkpoint; locate the per-Gaussian parameter tensors
import glob, torch, os
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'
CONFIG = sorted(glob.glob('/content/outputs/**/config.yml', recursive=True),
                key=os.path.getmtime)[-1]
CKPT = sorted(glob.glob(os.path.dirname(CONFIG) + '/nerfstudio_models/*.ckpt'))[-1]
state = torch.load(CKPT, map_location='cpu', weights_only=False)
pipe = state['pipeline']
gauss_keys = [k for k in pipe if 'gauss_params' in k]
means = pipe[[k for k in gauss_keys if k.endswith('means')][0]]
N = means.shape[0]
print(f'{N:,} Gaussians |', [k.split(".")[-1] for k in gauss_keys])

In [ ]:
# Selection A — geometric (uses known object placement; synthetic-only sanity check)
CENTER, RADIUS = torch.tensor([0.0, 1.2, 0.55]), 0.55
sel_geo = (means - CENTER).norm(dim=1) < RADIUS * 1.25
print(f'geometric: {sel_geo.sum().item():,} / {N:,}')

In [ ]:
# Selection B — mask lifting (real-scene capable): project every Gaussian centre
# into all cameras; select those inside the object mask in >90% of viewing cameras
import numpy as np, json
from PIL import Image
W, H = 832, 480
fx = 0.5 * W / np.tan(0.5 * meta['camera_angle_x'])
masks = torch.tensor(np.stack([
    np.array(Image.open(f"{root}/mask/{f['file_path'].split('/')[-1]}.png")) > 127
    for f in frames]))
w2cs = torch.tensor(np.stack([
    np.linalg.inv(np.array(f['transform_matrix'])) for f in frames]),
    dtype=torch.float32)

pts = torch.cat([means, torch.ones(N, 1)], dim=1)
inside = torch.zeros(N); visible = torch.zeros(N)
for i in range(len(w2cs)):
    p = (w2cs[i] @ pts.T).T
    x, y, z = p[:, 0], p[:, 1], p[:, 2]
    u = ( fx * x / -z + W / 2).long()          # OpenGL: camera looks down -Z
    v = (-fx * y / -z + H / 2).long()
    inb = (z < -1e-6) & (u >= 0) & (u < W) & (v >= 0) & (v < H)
    visible += inb.float()
    inside += (inb & masks[i][v.clamp(0, H-1), u.clamp(0, W-1)]).float()

sel_mask = (inside / visible.clamp(min=1) > 0.9) & (visible >= 40)
print(f'mask-lifted: {sel_mask.sum().item():,} | both: {(sel_geo & sel_mask).sum().item():,} '
      f'| only-geo: {(sel_geo & ~sel_mask).sum().item():,} '
      f'| only-mask: {(sel_mask & ~sel_geo).sum().item():,}')

In [ ]:
# Delete the union; save as a higher-step checkpoint so tools auto-load it
import copy
keep = ~(sel_geo | sel_mask)
state_del = copy.deepcopy(state)
for k in gauss_keys:
    state_del['pipeline'][k] = pipe[k][keep].clone()
state_del['step'] = state['step'] + 1
CKPT_DEL = os.path.join(os.path.dirname(CKPT), 'step-%09d.ckpt' % state_del['step'])
torch.save(state_del, CKPT_DEL)
print(f'kept {keep.sum().item():,} Gaussians ->', CKPT_DEL)

In [ ]:
# Render the deleted scene on held-out views
!TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1 ns-render dataset --load-config "$CONFIG" \
  --split test --output-path /content/renders_deleted

In [ ]:
# Visual check: object gone, footprint (mirror image, shadow, bleed) remains
import glob
from PIL import Image
import matplotlib.pyplot as plt
dele = sorted(glob.glob('/content/renders_deleted/test/rgb/*.jpg'))
test_ids = sorted(int(f['file_path'].split('/')[-1])
                  for i, f in enumerate(frames) if i in test_idx)
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].imshow(Image.open(dele[5])); ax[0].set_title('after deletion (M1)')
ax[1].imshow(Image.open(f'{DATA}/without/rgb/{test_ids[5]:04d}.png'))
ax[1].set_title('clean plate')
for a in ax: a.axis('off')
plt.show()

In [ ]:
# Score: PSNR vs clean plate, full-frame and inside the GT footprint region
import numpy as np, json
from PIL import Image

def load(p):
    return np.asarray(Image.open(p).convert('RGB').resize((W, H)),
                      dtype=np.float32) / 255.

def psnr(a, b, region=None):
    d = (a - b) ** 2
    if region is not None:
        if region.sum() == 0: return float('nan')
        d = d[region]
    return float(-10 * np.log10(d.mean() + 1e-12))

rows = []
for k, fid in enumerate(test_ids):
    gt_w, gt_c = load(f'{root}/rgb/{fid:04d}.png'), load(f'{DATA}/without/rgb/{fid:04d}.png')
    obj = np.array(Image.open(f'{root}/mask/{fid:04d}.png').resize((W, H))) > 127
    region = (np.abs(gt_w - gt_c).max(axis=2) > 0.05) & ~obj
    r = load(dele[k])
    rows.append({'frame': fid, 'psnr_full': psnr(r, gt_c),
                 'psnr_footprint': psnr(r, gt_c, region)})
avg = {k: float(np.nanmean([r[k] for r in rows])) for k in ('psnr_full', 'psnr_footprint')}
print(avg)
json.dump({'rows': rows, 'avg': avg}, open('/content/m1_baseline.json', 'w'), indent=2)

In [ ]:
import shutil, os
OUT = f'{DRIVE}/checkpoints/phase3_m1'
os.makedirs(OUT, exist_ok=True)
shutil.copy('/content/m1_baseline.json', f'{OUT}/m1_baseline.json')
shutil.copytree('/content/renders_deleted', f'{OUT}/renders_deleted', dirs_exist_ok=True)
print('saved to', OUT)